In [2]:
import numpy as np
import matplotlib.pyplot as plt
import illustris_python as il
import random
import h5py



In [3]:
snapnum = 40

In [4]:
basePath = '/cosma7/data/dp004/dc-zhan5/TNG300-1'

In [5]:
fields = ['GroupFirstSub', "GroupSFR", "GroupMass", "GroupNsubs", 
          "GroupPos", "GroupMassType", "GroupBHMass", "Group_M_TopHat200", "Group_M_Crit200"]
header = il.groupcat.loadHeader(f"{basePath}/output", snapnum)
halos = il.groupcat.loadHalos(f"{basePath}/output", snapnum, fields=fields)

In [6]:
h = 0.6774

In [7]:
mvir = np.log10(halos["Group_M_TopHat200"]*1e10)

/cosma/local/Python/3.6.5/lib/python3.6/site-packages/ipykernel_launcher.py:1: RuntimeWarning: divide by zero encountered in log10
  """Entry point for launching an IPython kernel.


In [8]:
group_sfr = np.log10(halos["GroupSFR"])

/cosma/local/Python/3.6.5/lib/python3.6/site-packages/ipykernel_launcher.py:1: RuntimeWarning: divide by zero encountered in log10
  """Entry point for launching an IPython kernel.


In [9]:
group_pos = halos["GroupPos"]/1e3

In [10]:
fields = ["SubhaloSFR", # [Msun/yr]
          "SubhaloGrNr",
         "SubhaloFlag",
         "SubhaloPos",
         "SubhaloCM", "SubhaloHalfmassRad", "SubhaloHalfmassRadType", "SubhaloMass", "SubhaloBHMass",
         "SubhaloMassType"] # [10^10 Msun/h]
print(len(fields))
subhalos = il.groupcat.loadSubhalos(f"{basePath}/output", snapnum, fields=fields)
print(type(subhalos))

10


<class 'dict'>


In [11]:
mask_mass = (mvir < 13.1) & (mvir > 13)

In [12]:
np.sum(mask_mass)

268

In [13]:
groupids = np.arange(0,len(mvir), 1)[mask_mass]

In [14]:
sub_pos = subhalos["SubhaloPos"]/1e3

In [15]:
group_pos1 = group_pos[mask_mass]

In [18]:
odir = "tng_toy_models/shuffled_sats_r=0-0.5/logM13.0-13.1/shuffle_individual/number_density"

ncount = 0

seed = 0
random.seed(seed)

nhalos = len(groupids)
ihalos = np.arange(0,nhalos)
halo_sfrs_dict = {i: 0 for i in range(nhalos + 1)}

with open(f"{odir}/sum_seed{seed}.txt", "w") as f_sum:
    with open(f"{odir}/gal_seed{seed}.txt", "w") as f:
        for ihalo, groupid in enumerate(groupids):
            mask_haloid = (subhalos["SubhaloGrNr"] == groupid) & (subhalos["SubhaloSFR"] > 0)
            pos_rel = sub_pos[mask_haloid] - group_pos[groupid]

            pos_rel_r = np.linalg.norm(pos_rel, axis=1)

            mask_r = (pos_rel_r > 0) & (pos_rel_r < 0.5) 
            #print(np.sum(mask_r))
            #print(len(pos_rel_r))
            #print(np.sum((subhalos["SubhaloGrNr"] == groupid)))
            ngal_in_halo = np.sum(mask_r)
            random_ihalos = np.random.choice(ihalos, ngal_in_halo)

            random_group_pos = [group_pos1[i] for i in random_ihalos]

            pos_arr_new = pos_rel[mask_r] + random_group_pos

            

            for igal in range(ngal_in_halo):
                    #halo_sfrs[ihalo] += 1
                    #halo_sfrs[shuffled_indices_list[ibin][ihalo]] += 1

                    #halo_sfrs_dict[random_ihalos[igal]] += subhalos["SubhaloSFR"][mask_haloid][mask_r][igal]

                    halo_sfrs_dict[random_ihalos[igal]] += 1
                    #print(groupid, 13, pos_arr_new[:,0][igal], pos_arr_new[:,1][igal], pos_arr_new[:,2][igal], np.log10(subhalos["SubhaloSFR"][mask_haloid][mask_r][igal]), file=f)
                    print(groupid, 13, pos_arr_new[:,0][igal], pos_arr_new[:,1][igal], pos_arr_new[:,2][igal], 1, file=f)
                   
          

            # mask_in = pos_rel_r[mask_r] < bin_edges_nonzero[0]
            # pos_arr_new = pos_rel[mask_r][mask_in] + shuffled_pos1_list[ibin][ihalo]

            # halo_sfrs[ihalo] += np.sum(subhalos["SubhaloSFR"][mask_haloid][mask_r][mask_in])

            # for igal in range(np.sum(mask_in)):
            #     print(groupid, 13, pos_arr_new[:,0][igal], pos_arr_new[:,1][igal], pos_arr_new[:,2][igal], np.log10(subhalos["SubhaloSFR"][mask_haloid][mask_r][mask_in][igal]), file=f)

            # ngal = np.sum(mask_r)

            # halo_sfr = np.sum(subhalos["SubhaloSFR"][mask_haloid][mask_r])
            
            # mask_test = pos_rel_r < 0.1
            # if np.sum(mask_test) > 1:
            #     ncount+=1
            #     print(np.sum(mask_test))
            # for igal in range(ngal):
            #     print(groupid, 13, pos_arr[:,0][igal], pos_arr[:,1][igal], pos_arr[:,2][igal], np.log10(subhalos["SubhaloSFR"][mask_haloid][mask_r][igal]), file=f)

    for ihalo, groupid in enumerate(groupids):
        #print(groupid, 13, group_pos1[:,0][ihalo], group_pos1[:,1][ihalo], group_pos1[:,2][ihalo], np.log10(halo_sfrs_dict[ihalo]), file=f_sum)
        print(groupid, 13, group_pos1[:,0][ihalo], group_pos1[:,1][ihalo], group_pos1[:,2][ihalo], halo_sfrs_dict[ihalo], file=f_sum)

    